# Data Cleansing

## Import all requirements

In [171]:
import pandas as pd
import numpy as np
import missingno as msgn

url = '../data/airbnb.csv'
airbnb = pd.read_csv(url)

## Price fixing


### Remove dollar sign

In [172]:
airbnb['price'] = airbnb['price'].str.replace('$', '', regex=False)

### Convert prices to numeric values FLOAT

In [173]:
airbnb['price'] = pd.to_numeric(airbnb['price'])
airbnb['price'] = airbnb['price'].fillna(airbnb['price'].median())

In [174]:
airbnb.rename(columns={'Unnamed: 0' : 'ID'}, inplace=True)

## Coordinates cleaning

In [175]:
coords = airbnb['coordinates'].str.slice(1, -1).str.partition(',').drop(columns=1)
airbnb['longitude'] = coords[0].astype('float')
airbnb['latitude'] = coords[2].astype('float')
airbnb.drop(columns='coordinates', inplace=True)

## Cleaning of the room types

In [176]:
airbnb['room_type'].unique()
for x in airbnb.index:
    if airbnb.loc[x, 'room_type'] == 'home':
        airbnb.loc[x, 'room_type'] = 'Entire home/apt'
    elif airbnb.loc[x, 'room_type'] == 'Private' or airbnb.loc[x, 'room_type'] == 'PRIVATE ROOM':
        airbnb.loc[x, 'room_type'] = 'Private room'
    elif airbnb.loc[x, 'room_type'] == '   Shared room      ':
        airbnb.loc[x, 'room_type'] = 'Shared room'

airbnb['room_type'].unique()

<StringArray>
['Private room', 'Entire home/apt', 'Shared room']
Length: 3, dtype: str

In [177]:
airbnb['listing_added'] = pd.to_datetime(airbnb['listing_added'])
airbnb.isna().any()

ID                    False
listing_id            False
name                   True
host_id               False
host_name              True
neighbourhood_full    False
room_type             False
price                 False
number_of_reviews     False
last_review            True
reviews_per_month      True
availability_365      False
rating                 True
number_of_stays        True
5_stars                True
listing_added         False
longitude             False
latitude              False
dtype: bool

## Mark not visited places

In [178]:
airbnb['last_review'] = pd.to_datetime(airbnb['last_review'])

airbnb['visited'] = ~airbnb['last_review'].isna()

## Replace reviews per month with 0 if there are no reviews

In [179]:
airbnb['reviews_per_month'] = airbnb['reviews_per_month'].fillna(airbnb['number_of_reviews'])

## Separate the neighbourhood

In [180]:
place = airbnb['neighbourhood_full'].str.slice(1, -1).str.partition(',').drop(columns=1)
airbnb['District'] = place[0]
airbnb['Neighborhood'] = place[2]
airbnb.drop(columns='neighbourhood_full', inplace=True)

## Fill the *Nan* house names with *House*

In [181]:
airbnb['name'] = airbnb['name'].fillna('House')

## Drop the airbnbs without owners

In [182]:
airbnb = airbnb.dropna(subset=['host_name'])

## Check if its okay

In [183]:
airbnb.info()

<class 'pandas.DataFrame'>
Index: 10017 entries, 0 to 10018
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ID                 10017 non-null  int64         
 1   listing_id         10017 non-null  int64         
 2   name               10017 non-null  str           
 3   host_id            10017 non-null  int64         
 4   host_name          10017 non-null  str           
 5   room_type          10017 non-null  str           
 6   price              10017 non-null  float64       
 7   number_of_reviews  10017 non-null  int64         
 8   last_review        7943 non-null   datetime64[us]
 9   reviews_per_month  10017 non-null  float64       
 10  availability_365   10017 non-null  int64         
 11  rating             7943 non-null   float64       
 12  number_of_stays    7943 non-null   float64       
 13  5_stars            7943 non-null   float64       
 14  listing_added      100

In [184]:
airbnb

,ID,listing_id,name,host_id,host_name,room_type,price,number_of_reviews,last_review,reviews_per_month,availability_365,rating,number_of_stays,5_stars,listing_added,longitude,latitude,visited,District,Neighborhood
0,0,13740704,"Cozy,budget friendly, cable inc, private entra...",20583125,Michel,Private room,45.0,10,2018-12-12,0.70,85,4.100954,12.0,0.609432,2018-06-08,40.63222,-73.93398,True,rooklyn,Flatland
1,1,22005115,Two floor apartment near Central Park,82746113,Cecilia,Entire home/apt,135.0,1,2019-06-30,1.00,145,3.367600,1.2,0.746135,2018-12-25,40.78761,-73.96862,True,anhattan,Upper West Sid
2,2,21667615,Beautiful 1BR in Brooklyn Heights,78251,Leslie,Entire home/apt,150.0,0,NaT,0.00,65,NaN,NaN,NaN,2018-08-15,40.70070,-73.99517,False,rooklyn,Brooklyn Height
3,3,6425850,"Spacious, charming studio",32715865,Yelena,Entire home/apt,86.0,5,2017-09-23,0.13,0,4.763203,6.0,0.769947,2017-03-20,40.79169,-73.97498,True,anhattan,Upper West Sid
4,4,22986519,Bedroom on the lively Lower East Side,154262349,Brooke,Private room,160.0,23,2019-06-12,2.29,102,3.822591,27.6,0.649383,2020-10-23,40.71884,-73.98354,True,anhattan,Lower East Sid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10014,10014,22307861,Lovely 1BR Harlem apartment,16004068,Rachel,Entire home/apt,105.0,4,2018-05-28,0.21,0,4.757555,4.8,0.639223,2017-11-22,40.80379,-73.95257,True,anhattan,Harle
10015,10015,953275,Apartment For Your Holidays in NYC!,4460034,Alain,Entire home/apt,125.0,50,2018-05-06,0.66,188,4.344704,60.0,0.648778,2017-10-31,40.79531,-73.93330,True,anhattan,East Harle
10016,10016,3452835,"Artsy, Garden Getaway in Central Brooklyn",666862,Amy,Entire home/apt,100.0,45,2016-11-27,0.98,0,3.966214,54.0,0.631713,2016-05-24,40.68266,-73.96743,True,rooklyn,Clinton Hil
10017,10017,23540194,"Immaculate townhouse in Clinton Hill, Brooklyn",67176930,Sophie,Entire home/apt,450.0,2,2019-05-31,0.17,99,4.078581,2.4,0.703360,2018-11-25,40.68832,-73.96366,True,rooklyn,Clinton Hil
